# Elecciones generales en la Comunidad de Madrid (1977-2019)

Resultados de las elecciones al Congreso por municipio en la Comunidad de Madrid:

1. Mapa del partido ganador en cada municipio, para cada convocatoria. Estos mapas son los que se publican en [`docs/elecciones_madrid/`](../docs/elecciones_madrid/).
2. Evolución del voto a cada partido en un municipio concreto.

**Datos necesarios** (ver [`data/README.md`](../data/README.md)):

- `raw/resultados_electorales/congreso_por_municipio/*.xlsx`: resultados por municipio (Ministerio del Interior).
- `raw/mapas/madrid_municipios.geojson`: contornos de los municipios de Madrid (IGN).

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

sys.path.insert(0, str(Path.cwd().parent))  # permite importar el paquete politicai desde notebooks/
from politicai.config import DOCS_DIR, RAW_DIR, STAGING_DIR, publicar_en_chart_studio
from politicai.electoral import COLORES_PARTIDOS, cargar_elecciones_municipio, columnas_partidos

pd.set_option("display.max_columns", None)

In [ ]:
CODIGO_PROVINCIA = 28  # Madrid (código INE de provincia; la Comunidad tiene una sola provincia)

ELECCIONES_DIR = RAW_DIR / "resultados_electorales" / "congreso_por_municipio"
ELECCIONES_COMPLETAS_PATH = STAGING_DIR / "elecciones_generales_completas.csv"
GEOJSON_PATH = RAW_DIR / "mapas" / "madrid_municipios.geojson"

## Datos electorales

Leer los Excel originales es lento, así que la tabla unida se guarda en `staging/` y se reutiliza en las siguientes ejecuciones (también la usa el notebook 02).

In [ ]:
if ELECCIONES_COMPLETAS_PATH.exists():
    df = pd.read_csv(ELECCIONES_COMPLETAS_PATH, sep="|", dtype={"CODIGOINE": str}, index_col=0)
else:
    df = cargar_elecciones_municipio(ELECCIONES_DIR)
    ELECCIONES_COMPLETAS_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(ELECCIONES_COMPLETAS_PATH, sep="|")

party_columns = columnas_partidos(df)
df[party_columns] = df[party_columns].apply(pd.to_numeric)
df.shape

In [ ]:
df_comunidad = df[df["codigo_de_provincia"] == CODIGO_PROVINCIA].copy()
df_comunidad.shape

In [ ]:
with open(GEOJSON_PATH) as geofile:
    geojson = json.load(geofile)

## Partido ganador por municipio

Se descartan los municipios sin votos registrados en una elección (suele tratarse de municipios que aún no existían).

In [ ]:
df_comunidad["ganador"] = df_comunidad[party_columns].idxmax(axis=1)
df_comunidad = df_comunidad[df_comunidad[party_columns].sum(axis=1) > 0]
df_comunidad["ganador"].unique()

In [ ]:
df_ganadores = df_comunidad[["ganador", "nombre_de_municipio", "CODIGOINE", "yearmonth"]].copy()
df_ganadores["yearmonth"] = df_ganadores["yearmonth"].astype(str)
ELECCIONES = sorted(df_ganadores["yearmonth"].unique())
ELECCIONES

In [ ]:
def mapa_ganadores(yearmonth):
    """Mapa coroplético con el partido ganador en cada municipio para una elección (AAAAMM)."""
    datos = df_ganadores[df_ganadores["yearmonth"] == yearmonth]
    return px.choropleth_map(
        data_frame=datos,
        geojson=geojson,
        locations="CODIGOINE",
        featureidkey="properties.CODIGOINE",
        color="ganador",
        color_discrete_map=COLORES_PARTIDOS,
        hover_data=["nombre_de_municipio", "ganador"],
        map_style="carto-positron",
        zoom=8,
        center={"lat": 40.53, "lon": -3.75},
        height=800,
        title=f"Elecciones Generales {yearmonth[:4]}-{yearmonth[4:]} en Madrid",
    )


mapa_ganadores("201911").show()

Exportar un mapa HTML por elección a `docs/elecciones_madrid/` (es lo que se sirve en GitHub Pages):

In [ ]:
EXPORTAR_MAPAS = False

if EXPORTAR_MAPAS:
    salida = DOCS_DIR / "elecciones_madrid"
    for yearmonth in ELECCIONES:
        mapa_ganadores(yearmonth).write_html(salida / f"elecciones_madrid_{yearmonth}.html")

## Evolución del voto en un municipio

Votos de los 15 partidos más votados en el municipio a lo largo de todas las elecciones. Las coaliciones y refundaciones se agrupan bajo un mismo nombre (AP -> PP, Unidas Podemos -> Podemos...).

In [ ]:
CODIGOINE_MUNICIPIO = "28092"  # Móstoles

df_municipio = df_comunidad[df_comunidad["CODIGOINE"] == CODIGOINE_MUNICIPIO]
nombre_municipio = df_municipio["nombre_de_municipio"].unique()[0]
df_municipio.head()

In [ ]:
partidos_con_votos = [p for p in party_columns if df_municipio[p].sum() > 0]
df_party_series = df_municipio.melt(id_vars=["year", "month"], value_vars=partidos_con_votos)
df_party_series = df_party_series.rename(columns={"variable": "partido", "value": "votos"})

# Se excluye abril de 2019 para tener una única elección por año en el eje X.
df_party_series = df_party_series[~((df_party_series["year"] == 2019) & (df_party_series["month"] == 4))]

top_parties = df_party_series.groupby("partido")["votos"].sum().nlargest(15).index
df_party_series = df_party_series[df_party_series["partido"].isin(top_parties)]
df_party_series["partido"] = df_party_series["partido"].replace({
    "ap_pdp_pl": "ap", "ap_pdp": "ap",
    "podemos_iu": "podemos", "podemos_iu_equo": "podemos",
    "iu_lv": "iu",
})
df_party_series = df_party_series.groupby(["year", "month", "partido"], as_index=False)["votos"].sum()
df_party_series["size"] = 5  # tamaño fijo de los puntos en el gráfico animado

In [ ]:
fig_town_time = px.scatter(
    df_party_series, x="year", y="votos", color="partido",
    animation_frame="year", animation_group="partido", size="size",
    range_x=[1976, 2020], range_y=[0, int(df_party_series["votos"].max() * 1.1)],
    color_discrete_map=COLORES_PARTIDOS, template="plotly_white", height=600,
    title=f"Elecciones Generales a lo largo del tiempo en {nombre_municipio}",
)
fig_town_time.show()

In [ ]:
fig_town_lines = px.line(
    df_party_series, x="year", y="votos", color="partido", line_group="partido", hover_name="partido",
    range_x=[1976, 2020], color_discrete_map=COLORES_PARTIDOS, template="plotly_white",
    title=f"Elecciones Generales a lo largo del tiempo en {nombre_municipio}",
)
fig_town_lines.show()

In [ ]:
# publicar_en_chart_studio(fig_town_time, "elecciones_generales_mostoles_tiempo")